# 00 - Preparación de bases EPH

Este notebook toma las bases de la EPH descargadas manualmente del INDEC y subidas a
**Google Drive** (carpeta `carga_EPH`), une la base de **individuos** con la de
**hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`), y deja los datasets
listos en `data/processed/` para que los usen el resto de los notebooks.

**Salidas:**
- `data/processed/eph_T<Q><YY>.parquet`: un archivo por trimestre, ya con individuos+hogares unidos.
- `data/processed/eph_panel.parquet`: panel con todos los trimestres concatenados (con columnas `ANIO` y `TRIMESTRE`).

**Cómo agregar un trimestre nuevo:**
1. Descargar del INDEC el `.zip` del trimestre (ej. `EPH_usu_4_Trim_2025_txt.zip`),
   que contiene `usu_individual_T<Q><YY>.txt` y `usu_hogar_T<Q><YY>.txt`.
2. Subir el `.zip` (sin descomprimir) a Google Drive, carpeta `carga_EPH`.
3. Volver a correr este notebook: detecta automáticamente todos los trimestres
   presentes en `carga_EPH` (no hace falta editar ninguna lista).

## Setup (Colab)

Clona el repo para tener acceso a `src/data_loader.py`, y monta Google Drive para
leer los `.zip` subidos a `carga_EPH`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "analisis_EPH"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}
sys.path.append(os.getcwd())

from google.colab import drive
drive.mount('/content/drive')

## Chequeo de disponibilidad

Escanea `/content/drive/MyDrive/carga_EPH` (y `data/raw/` del repo) buscando los `.zip`
del INDEC, y lista los trimestres detectados (con ambas bases, individual y hogar).

In [ ]:
from src.data_loader import list_available_quarters, _period_tag

available = list_available_quarters()
print("Disponibles:", [_period_tag(y, p) for y, p in available])

## Construcción del panel

Une individuos + hogares por trimestre (`CODUSU` + `NRO_HOGAR`), agrega `ANIO`/`TRIMESTRE`, y guarda en `data/processed/`.

In [ ]:
from src.data_loader import build_panel

panel = build_panel(available, save=True)
panel.shape

In [ ]:
panel[["ANIO", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "COMPONENTE"]].head()

## Subir resultados a GitHub (opcional)

Los archivos `.parquet` generados en `data/processed/` quedan disponibles para los demás notebooks vía `raw.githubusercontent.com` una vez que se comiteen y pusheen al repo (esto se hace desde el entorno local, no desde Colab).